# Notebook 2: RDF Data & SHACL Validation

Now that we have the ontology, we populate it with instance data and validate it using SHACL shapes - first for **data quality** (structural), then for **business rules** (eligibility).


In [ ]:
from workshop import *

---
## Module 2: Instance Data

The data file contains **5 sample claims** across **3 policies**, each demonstrating a different scenario:

| Claim | Policy | Amount | Status | Scenario |
|-------|--------|--------|--------|----------|
| C001 | P001 (FSA, Nora Copay) | 5,000 | Approved then Paid | Normal lifecycle |
| C002 | P001 (FSA, Nora Copay) | 1,200 | Submitted | Pending |
| C003 | P002 (HSA, Owen Deductible) | 3,000 | Denied | Coverage gap (SCD2 temporal) |
| C004 | P003 (HRA, Lily Referral) | 15,000 | Denied | Exceeds net coverage limit |
| C005 | P001 (FSA, Nora Copay) | 4,000 | Denied | Filed after 90-day window |


In [ ]:
# Display the data TTL
data_ttl = (ONTOLOGY_DIR / "data.ttl").read_text()
display(Markdown(f"```turtle\n{data_ttl}\n```"))

Now we load the ontology and instance data into an in-memory graph and verify the data by listing all claims. This confirms the data conforms to the ontology structure and gives us a baseline before validation:


In [ ]:
# Load ontology (always local) + claim data (Ontop or fallback)
from rdflib import Graph, Namespace, RDF, RDFS, OWL

g = Graph()
g.parse(str(ONTOLOGY_DIR / "ontology.ttl"))

data_graph, data_source = load_claim_data()
g += data_graph

print(f"Data source: {data_source}")
print(f"Combined graph: {len(g)} triples")

# Quick verification — list all claims
results = g.query("""
    PREFIX : <http://example.org/insurance#>
    SELECT ?claimId ?amount ?incidentDate ?policyId
    WHERE {
        ?claim a :Claim ;
               :claimId ?claimId ;
               :claimAmount ?amount ;
               :incidentDate ?incidentDate ;
               :forPolicy ?policy .
        ?policy :policyId ?policyId .
    }
    ORDER BY ?claimId
""")

print(f"\n{'Claim':<8} {'Policy':<8} {'Amount':>10} {'Incident Date'}")
print("─" * 50)
for row in results:
    print(f"{row.claimId:<8} {row.policyId:<8} ${float(row.amount):>9,.2f} {str(row.incidentDate)[:10]}")

---
## Module 3: SHACL Structural Shapes (Data Quality)

These shapes validate **data quality** - does every claim have an ID? Is every amount non-negative?  
They catch *structural* problems, not *business logic* problems.

**4 shapes**: `ClaimShape`, `PolicyShape`, `PolicyCoverageShape`, `ClaimEventShape`

In [ ]:
# Display the structural shapes
shapes_ttl = (ONTOLOGY_DIR / "shapes.ttl").read_text()
display(Markdown(f"```turtle\n{shapes_ttl}\n```"))

Run the SHACL structural validation against our loaded data. If the data is well-formed - every required field present, correct types, valid enumerations - this should pass cleanly:


In [ ]:
# Run structural validation
from pyshacl import validate

shapes_graph = Graph()
shapes_graph.parse(str(ONTOLOGY_DIR / "shapes.ttl"))

conforms, results_graph, results_text = validate(
    g,
    shacl_graph=shapes_graph,
    inference='rdfs',
    abort_on_first=False,
)

print(f"Data quality validation: {'✅ PASS — all structural shapes conform' if conforms else '❌ FAIL'}")
if not conforms:
    print(f"\nViolations:\n{results_text}")

---
## Module 4: SHACL Eligibility Shapes (Business Rules)

This is where semantic reasoning gets practical. These shapes encode **business rules** as SPARQL queries:

| Shape | Business Rule | What It Catches |
|-------|--------------|----------------|
| `ClaimHasActiveCoverageShape` | Coverage must be active at incident date | SCD2 temporal gaps |
| `ClaimWithinCoverageLimitShape` | Claim ≤ coverage − deductible | Over-limit claims |
| `ClaimFiledWithinWindowShape` | Filed within 90 days of incident | Late submissions |
| `ClaimPolicyUnambiguousCoverageShape` | Only one current coverage per policy | SCD2 data integrity |

When a claim **fails** these shapes, the SHACL violation report becomes the machine-readable "denial reason" - no hardcoded denial codes needed.

In [ ]:
# Display the eligibility shapes
elig_ttl = (ONTOLOGY_DIR / "eligibility_shapes.ttl").read_text()
display(Markdown(f"```turtle\n{elig_ttl}\n```"))

Now run the eligibility shapes against the same data. Unlike structural validation, we **expect violations here** - these are the denied claims whose denial reasons are derived by the rules:


In [ ]:
# Run eligibility validation — this is where denials are detected
elig_shapes = Graph()
elig_shapes.parse(str(ONTOLOGY_DIR / "eligibility_shapes.ttl"))

conforms, results_graph, results_text = validate(
    g,
    shacl_graph=elig_shapes,
    inference='rdfs',
    abort_on_first=False,
)

print(f"Eligibility validation: {'✅ All claims eligible' if conforms else '⚠️  Eligibility violations found'}")
print(f"\n{results_text}")

Let's parse the SHACL validation report into a structured format so we can see exactly which claims violated which rules, and why:


In [ ]:
# Parse the violations into a structured dict keyed by claim URI
from rdflib import Namespace as NS

SH = NS("http://www.w3.org/ns/shacl#")

violations_by_claim = {}
for result in results_graph.subjects(RDF.type, SH.ValidationResult):
    focus = str(results_graph.value(result, SH.focusNode))
    message = str(results_graph.value(result, SH.resultMessage))
    source = str(results_graph.value(result, SH.sourceShape))
    
    violations_by_claim.setdefault(focus, []).append({
        "message": message,
        "source_shape": source.replace(str(INS), ":"),
    })

print(f"Claims with violations: {len(violations_by_claim)}\n")
for uri, viols in sorted(violations_by_claim.items()):
    claim_id = uri.split("#")[-1].replace("claim", "C").replace("00", "00")
    print(f"  {claim_id} ({len(viols)} violation{'s' if len(viols) > 1 else ''})")
    for v in viols:
        print(f"    Shape: {v['source_shape']}")
        print(f"    → {v['message'][:120]}")
    print()

---

## Module 4.5: Deduction - Deriving Net Payable via SPARQL CONSTRUCT

So far, every property in our graph is **asserted** - it comes directly from the source data.
But some facts can be **derived** by reasoning over existing triples.

`:netPayable` is a derived property: the amount the insurer actually owes after applying the deductible.
It doesn't exist in `data.ttl` - we compute it with a SPARQL `CONSTRUCT` that:

1. Finds claims with a `paid` event
2. Matches the policy coverage active at the incident date (SCD2 temporal join)
3. Computes `min(paidAmount, coverageAmount - deductible)`
4. Materializes the result as new `:netPayable` triples in the graph

This is the simplest form of **semantic deduction** - new knowledge derived from existing knowledge using rules.
The same pattern scales to OWL reasoning, SWRL rules, or SHACL Advanced Features (`sh:rule`).


In [ ]:
# ── Derive :netPayable via SPARQL CONSTRUCT ─────────────────────────────
# This query creates NEW triples that don't exist in the source data.
# It reasons across Claim -> Policy -> PolicyCoverage (temporal match)
# to compute the insurer's liability after the deductible.

CONSTRUCT_NET_PAYABLE = """
PREFIX : <http://example.org/insurance#>
CONSTRUCT {
    ?claim :netPayable ?payout .
}
WHERE {
    ?claim a :Claim ;
           :incidentDate ?incidentDate ;
           :forPolicy ?policy ;
           :hasEvent ?event .
    ?event :eventType "paid" ;
           :eventAmount ?paidAmount .

    # Temporal join: find coverage active at incident date
    ?policy :hasCoverage ?cov .
    ?cov :coverageAmount ?coverage ;
         :deductible ?deductible ;
         :validFrom ?from .
    OPTIONAL { ?cov :validTo ?to }
    FILTER (?from <= ?incidentDate)
    FILTER (!BOUND(?to) || ?to > ?incidentDate)

    # Net payable = min(paid amount, coverage - deductible)
    BIND(?coverage - ?deductible AS ?netCoverage)
    BIND(IF(?paidAmount > ?netCoverage, ?netCoverage, ?paidAmount) AS ?payout)
}
"""

# Execute the CONSTRUCT — returns a graph of derived triples
result = g.query(CONSTRUCT_NET_PAYABLE)
derived_graph = result.graph

print(f"Derived {len(derived_graph)} new triple(s) via CONSTRUCT:\n")
for s, p, o in derived_graph:
    claim_id = str(s).split('#')[-1]
    print(f"  {claim_id}  :netPayable  ${float(o):,.2f}")

# Add derived triples to the main graph
before = len(g)
g += derived_graph
print(f"\nGraph: {before} -> {len(g)} triples (+{len(g) - before} derived)")


→ Proceed to **03-Neptune**
